# V14 · 前20小时突破支持度复核

## tl;dr
三格普通Python执行完成：原病例251个，符合60、不满足191、未知0；
原控制462个，符合53、不满足409、未知0。
支持状态：insufficient_support_no_outcomes。没有收益回放或盈利结论。
复用同一固定验证器，不是第二独立验证；Jupyter与完整schema验证未完成。

## Context & Methods
多头K1收盘严格高于自身前20根完整小时高点，空头严格低于前20小时低点。
K1自身不得混入前20根极值；等于边界为不满足，缺小时或断档保留未知。
原母事件及控制完整保留，不用后来涨跌、最大浮盈或成交结果挑选支持。

### Key Assumptions
保存窗口只包含入场前已完成信息。验证器重算保存小时的极值、时钟和支持计数，
不能独立证明原始5m到小时聚合正确。下面复用同一已固定stdlib验证器，
不是第二个独立实现，不执行它的整仓扫描或金融审查入口。
2023–2024为重复开发期；支持数不是盈利能力、统计功效或新鲜独立证据。

## Data
只读取固定summary和四份保存CSV：entry_context、counts、matched_support、prior_hourly_rows。
summary字节SHA固定，四份CSV需与其output_hashes一致。两个验证脚本均固定SHA，
禁止执行未固定的替代脚本。未读取原始归档，不读取V5/V13等历史收益表。
从仓库运行，或设置NOTEBOOK_REPOSITORY_ROOT。代码格只需Python标准库。

In [1]:
import csv, hashlib, importlib.util, io, json, re
from pathlib import Path
RESULTS_RELATIVE='experiments/active/exp-btcusdtp-1h-prior20-breakout-preholdout-20260906-v14/results'
EVIDENCE_FILES=('entry_context.csv', 'counts.csv', 'matched_support.csv', 'prior_hourly_rows.csv')
VERIFIER_FILES=('scripts/verify_hourly_impulse_prior_breakout_v14.py', 'scripts/verify_hourly_impulse_launch_v11.py')
SUMMARY_SHA256='1a6f1d64ec4448d756a37f659346cea26486c1e7923eb3998985fb5a793924f4'
VERIFIER_HASHES={'scripts/verify_hourly_impulse_prior_breakout_v14.py': '0227ecfd4ab59464eaf5a9b9a19b353ec2a2cc6440f9ffa082b1fa4d6d2e9ef7', 'scripts/verify_hourly_impulse_launch_v11.py': '2b12c8309301bf2c7960679838ee048c30960a353317a2454842f2ad6c892362'}
def require(condition,message):
    if not condition:raise ValueError(message)
def digest(data):return hashlib.sha256(data).hexdigest()
hint=globals().get("NOTEBOOK_REPOSITORY_ROOT")
roots=[Path(hint)] if hint is not None else [Path.cwd(),*Path.cwd().parents]
root=next((p.resolve() for p in roots if (p/RESULTS_RELATIVE/"summary.json").is_file()),None)
require(root is not None,"Run from repository or set NOTEBOOK_REPOSITORY_ROOT")
directory=(root/RESULTS_RELATIVE).resolve()
require(directory.is_relative_to(root),"Evidence directory escaped repository")
def evidence_path(name):
    require(name in ("summary.json",*EVIDENCE_FILES),"File not in support allowlist")
    path=(directory/name).resolve()
    require(path==directory/name,"Support evidence symlink escaped fixed identity")
    return path
def verifier_path(name):
    require(name in VERIFIER_FILES,"Verifier not allowlisted")
    path=(root/name).resolve()
    require(path==root/name,"Verifier symlink changed identity")
    return path
print("Support-only saved inputs:",RESULTS_RELATIVE)

Support-only saved inputs: experiments/active/exp-btcusdtp-1h-prior20-breakout-preholdout-20260906-v14/results


### 1. 固定摘要、四份CSV和验证器依赖

In [2]:
payload=evidence_path("summary.json").read_bytes()
require(digest(payload)==SUMMARY_SHA256,"Pinned summary hash mismatch")
def reject_constant(value):raise ValueError("Nonfinite JSON: "+value)
summary=json.loads(payload,parse_constant=reject_constant)
require(summary["experiment_id"]=='exp-btcusdtp-1h-prior20-breakout-preholdout-20260906-v14',"Wrong experiment")
require(summary["status"] in ("insufficient_support_no_outcomes","support_pass_requires_separate_replay"),"Not support-only")
for flag in ("outcomes_read_or_computed","profitability_test","holdout_consumed","training_eligible","production_eligible"):
    require(summary[flag] is False,"Unexpected financial/production claim: "+flag)
require(type(summary["outcome_replays"]) is int and summary["outcome_replays"]==0,"Outcome replay is not support evidence")
forbidden=re.compile(r"(^|_)(pnl|returns?|mfe|mae|outcome|closed|exit|profit|loss|fee)($|_)",re.I)
tables={}
for name in EVIDENCE_FILES:
    data=evidence_path(name).read_bytes()
    require(digest(data)==summary["output_hashes"][name],"CSV hash mismatch: "+name)
    reader=csv.DictReader(io.StringIO(data.decode("utf-8")))
    columns=reader.fieldnames
    require(columns and len(columns)==len(set(columns)),"Missing or duplicate CSV header")
    require(not any(forbidden.search(c) or c.startswith("max_favourable") for c in columns),"Economic columns forbidden in support notebook")
    rows=list(reader)
    require(all(None not in r and all(v is not None for v in r.values()) for r in rows),"Malformed support CSV")
    tables[name]=rows
for name in VERIFIER_FILES:
    require(digest(verifier_path(name).read_bytes())==VERIFIER_HASHES[name],"Verifier dependency hash mismatch: "+name)
print("Pinned summary, four saved CSVs and both verifier modules verified")

Pinned summary, four saved CSVs and both verifier modules verified


## Results

### 2. 复用固定验证器重算支持度，不运行收益回放

In [3]:
spec=importlib.util.spec_from_file_location("_v14_notebook_saved_verifier",verifier_path(VERIFIER_FILES[0]))
verifier=importlib.util.module_from_spec(spec)
spec.loader.exec_module(verifier)
validation=verifier.verify_tables(tables["entry_context.csv"],tables["prior_hourly_rows.csv"],
    tables["counts.csv"],tables["matched_support.csv"],summary)
require(isinstance(validation,dict) and validation.get("status")=="passed","Verifier must return a passed validation receipt")
verified={"status":summary["status"],"population":summary["population"],
    "support_values":summary["support_values"],"support_gates":summary["support_gates"],
    "support_pass":summary["support_pass"],"matching":summary["matching"],"outcome_replays":0}
print("Verified saved support:",json.dumps(verified,ensure_ascii=False,allow_nan=False))
print("SAME pinned stdlib verifier reused; not independent raw aggregation or financial validation.")

Verified saved support: {"status": "insufficient_support_no_outcomes", "population": {"case": {"total": 251, "accepted": 60, "abstain": 191, "unknown": 0}, "control": {"total": 462, "accepted": 53, "abstain": 409, "unknown": 0}}, "support_values": {"events": 60, "minimum_fold_events": 11, "active_months": 23, "minimum_fold_months": 5}, "support_gates": {"minimum_events": false, "minimum_per_fold": false, "minimum_active_months": true, "minimum_months_per_fold": true}, "support_pass": false, "matching": {"matched": 154, "unmatched": 97, "all_known": 154, "coverage": 0.6135458167330677, "required_coverage": 0.9, "coverage_pass": false}, "outcome_replays": 0}
SAME pinned stdlib verifier reused; not independent raw aggregation or financial validation.


## Takeaways
支持不足不能通过看收益后缩小窗口、放松匹配或选择盈利子集来补足。
即使支持门通过，也只是允许另立回放；这里没有获利、手续费、止盈止损结果或生产资格。
保留未知和原154/251匹配覆盖限制。保存小时窗口复核不等于独立原始行情重建。

### Execution gap
Plain Python top-down execution is not Jupyter-kernel execution. Minimum nbformat4.5 structure and code compilation are checked; full nbformat schema validation is not run. nbformat, nbclient and ipykernel are unavailable; no dependencies were installed.

完整Jupyter验证需在已有依赖的隔离环境运行
`python -m jupyter nbconvert --execute --to notebook --inplace path/to/prior_breakout_support.ipynb`。
本轮不安装依赖；三格普通Python执行不冒充Jupyter或完整schema验证。